# 03C. Signal Decay Engine

Measure whether candidate signals' predictive IC is stable, improving, or decaying over time. This notebook adds diagnostic evidence only; it does not approve signals, replace WFV, stress test, freeze, portfolio construct, or use ML.

## 1. Purpose and scope

Use existing candidate signal and scoring artifacts to compute rolling cross-sectional IC curves and decay summaries. Existing scoring, WFV, stress, freeze, composite, and portfolio tables are not modified.

## 2. Imports and config

In [1]:
from __future__ import annotations

import sys
from pathlib import Path

import pandas as pd
from IPython.display import display

cwd = Path.cwd().resolve()
project_root = next((path for path in [cwd, *cwd.parents] if (path / 'src').exists()), None)
if project_root is None:
    raise RuntimeError('Could not locate project root from current working directory.')

if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src.db import get_db_path, load_price_table, load_table
from src.run_config import make_run_id, make_run_timestamp
from src.signal_decay import run_signal_decay_analysis
from src.signal_decay_storage import SIGNAL_DECAY_TABLES, save_signal_decay_outputs
from src.signal_storage import load_candidate_signals

DECAY_VERSION = 'phase2_signal_decay_v1'
ROLLING_IC_WINDOW = 63
MIN_ROLLING_OBS = 8
HORIZONS = [1, 5, 10, 20]
IC_METHOD = 'spearman'

sqlite_db_path = get_db_path()

print(f'SQLite database: {sqlite_db_path}')
print(f'Decay version: {DECAY_VERSION}')

SQLite database: /Users/AnyiXu_1/Desktop/multi-factor-equity-alpha-model/sql/project_underdog.db
Decay version: phase2_signal_decay_v1


## 3. Create decay run_id / timestamp

In [2]:
run_id = make_run_id(prefix='phase2_nb03c_signal_decay')
run_timestamp = make_run_timestamp()

print(f'run_id: {run_id}')
print(f'run_timestamp: {run_timestamp}')

run_id: phase2_nb03c_signal_decay_20260510_173231
run_timestamp: 2026-05-10 17:32:31


## 4. Load candidate_signals_current

In [3]:
candidate_signals = load_candidate_signals(current=True, db_path=sqlite_db_path)

candidate_signal_summary = pd.DataFrame(
    [
        {
            'rows': len(candidate_signals),
            'n_signals': candidate_signals['signal_name'].nunique(),
            'n_tickers': candidate_signals['ticker'].nunique(),
            'start_date': candidate_signals['Date'].min(),
            'end_date': candidate_signals['Date'].max(),
        }
    ]
)

display(candidate_signal_summary)

,rows,n_signals,n_tickers,start_date,end_date
0,108307152,108,478,2018-01-02,2026-05-07


## 5. Load signal scoring artifacts

In [4]:
signal_scores = load_table('signal_scores_current', db_path=sqlite_db_path)
signal_best_horizon = load_table('signal_best_horizon_current', db_path=sqlite_db_path)
signal_scoring_gate = load_table('signal_scoring_gate_current', db_path=sqlite_db_path)

display(signal_scores.head())
display(signal_best_horizon.head())
display(signal_scoring_gate.head())

,signal_name,horizon,method,n_obs,mean_ic,median_ic,ic_std,ic_ir,hit_rate,positive_ic_rate,missing_pct,signal_family,signal_version,run_id,scoring_version
0,close_position_reversal_5,1,spearman,600770,0.006244,0.000382,0.190790,0.032728,0.499689,0.500491,0.400934,microstructure_lite,phase2_orthogonal_signals_v2,signal_scoring_20260510_222922,phase2_signal_scoring_v2
1,close_position_reversal_5,5,spearman,589206,0.007745,0.003505,0.179986,0.043031,0.496022,0.508358,0.412465,microstructure_lite,phase2_orthogonal_signals_v2,signal_scoring_20260510_222922,phase2_signal_scoring_v2
2,close_position_reversal_5,10,spearman,578401,0.006744,0.008758,0.173888,0.038786,0.494546,0.517003,0.423239,microstructure_lite,phase2_orthogonal_signals_v2,signal_scoring_20260510_222922,phase2_signal_scoring_v2
3,close_position_reversal_5,20,spearman,562984,0.008850,0.009159,0.166689,0.053094,0.494270,0.526994,0.438613,microstructure_lite,phase2_orthogonal_signals_v2,signal_scoring_20260510_222922,phase2_signal_scoring_v2
4,dollar_volume_shock_20,1,spearman,594678,0.002990,0.004823,0.092843,0.032205,0.500327,0.523360,0.407008,liquidity_flow,phase2_orthogonal_signals_v2,signal_scoring_20260510_222922,phase2_signal_scoring_v2


,signal_name,signal_family,best_horizon,best_mean_ic,best_abs_mean_ic,best_ic_ir,best_positive_ic_rate,best_hit_rate,signal_direction,signal_strength,run_id,scoring_version
0,expanded_reversal_1d,mean_reversion,1,0.017276,0.017276,0.084684,0.529553,0.505222,POSITIVE_EDGE,WEAK,signal_scoring_20260510_222922,phase2_signal_scoring_v2
1,expanded_residual_market_return_20,residual_relative_value,20,0.016748,0.016748,0.088930,0.528145,0.501801,POSITIVE_EDGE,WEAK,signal_scoring_20260510_222922,phase2_signal_scoring_v2
2,vol_of_vol_20,volatility_structure,20,0.015976,0.015976,0.128788,0.553403,0.501826,POSITIVE_EDGE,WEAK,signal_scoring_20260510_222922,phase2_signal_scoring_v2
3,expanded_reversal_3d,mean_reversion,1,0.014939,0.014939,0.070354,0.521886,0.503985,POSITIVE_EDGE,WEAK,signal_scoring_20260510_222922,phase2_signal_scoring_v2
4,range_expansion_failure_5,volatility_structure,20,0.014333,0.014333,0.112310,0.543756,0.497760,POSITIVE_EDGE,WEAK,signal_scoring_20260510_222922,phase2_signal_scoring_v2


,signal_name,horizon,method,n_obs,mean_ic,median_ic,ic_std,ic_ir,hit_rate,positive_ic_rate,...,signal_family,signal_version,abs_mean_ic,abs_ic_ir,signal_direction,signal_strength,status,scoring_gate_notes,run_id,scoring_version
0,close_position_reversal_5,1,spearman,600770,0.006244,0.000382,0.190790,0.032728,0.499689,0.500491,...,microstructure_lite,phase2_orthogonal_signals_v2,0.006244,0.032728,POSITIVE_EDGE,NO_SIGNAL,REJECTED_LOW_SIGNAL,Fails preliminary predictive scoring thresholds.,signal_scoring_20260510_222922,phase2_signal_scoring_v2
1,close_position_reversal_5,5,spearman,589206,0.007745,0.003505,0.179986,0.043031,0.496022,0.508358,...,microstructure_lite,phase2_orthogonal_signals_v2,0.007745,0.043031,POSITIVE_EDGE,NO_SIGNAL,REJECTED_LOW_SIGNAL,Fails preliminary predictive scoring thresholds.,signal_scoring_20260510_222922,phase2_signal_scoring_v2
2,close_position_reversal_5,10,spearman,578401,0.006744,0.008758,0.173888,0.038786,0.494546,0.517003,...,microstructure_lite,phase2_orthogonal_signals_v2,0.006744,0.038786,POSITIVE_EDGE,NO_SIGNAL,REJECTED_LOW_SIGNAL,Fails preliminary predictive scoring thresholds.,signal_scoring_20260510_222922,phase2_signal_scoring_v2
3,close_position_reversal_5,20,spearman,562984,0.008850,0.009159,0.166689,0.053094,0.494270,0.526994,...,microstructure_lite,phase2_orthogonal_signals_v2,0.008850,0.053094,POSITIVE_EDGE,NO_SIGNAL,REJECTED_LOW_SIGNAL,Fails preliminary predictive scoring thresholds.,signal_scoring_20260510_222922,phase2_signal_scoring_v2
4,dollar_volume_shock_20,1,spearman,594678,0.002990,0.004823,0.092843,0.032205,0.500327,0.523360,...,liquidity_flow,phase2_orthogonal_signals_v2,0.002990,0.032205,POSITIVE_EDGE,NO_SIGNAL,REJECTED_LOW_SIGNAL,Fails preliminary predictive scoring thresholds.,signal_scoring_20260510_222922,phase2_signal_scoring_v2


## 6. Load clean close prices

In [5]:
close_prices = load_price_table('clean_close_prices_current', db_path=sqlite_db_path)

display(pd.DataFrame([{'rows': close_prices.shape[0], 'columns': close_prices.shape[1]}]))
display(close_prices.head())

,rows,columns
0,2098,478


,A,AAPL,ABBV,ABNB,ABT,ACGL,ACN,ADBE,ADI,ADM,...,WTW,WY,WYNN,XEL,XOM,XYL,YUM,ZBH,ZBRA,ZTS
Date,,,,,,,,,,,,,,,,,,,,,
2018-01-02,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2018-01-03,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2018-01-04,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2018-01-05,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2018-01-08,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## 7. Run rolling IC / decay analysis

In [6]:
decay_curve, decay_summary = run_signal_decay_analysis(
    signal_scores=signal_scores,
    candidate_signals_long=candidate_signals,
    close_prices=close_prices,
    horizons=HORIZONS,
    window=ROLLING_IC_WINDOW,
    method=IC_METHOD,
    min_rolling_obs=MIN_ROLLING_OBS,
)

display(pd.DataFrame([{'decay_curve_rows': len(decay_curve), 'decay_summary_rows': len(decay_summary)}]))
display(decay_curve.head())

approved_signal_long duplicate diagnostics for ['signal_name', 'Date', 'ticker']: duplicate_rows=0, duplicate_groups=0
approved_signal_long duplicate diagnostics for ['signal_name', 'Date', 'ticker']: duplicate_rows=0, duplicate_groups=0
approved_signal_long duplicate diagnostics for ['signal_name', 'Date', 'ticker']: duplicate_rows=0, duplicate_groups=0
approved_signal_long duplicate diagnostics for ['signal_name', 'Date', 'ticker']: duplicate_rows=0, duplicate_groups=0
approved_signal_long duplicate diagnostics for ['signal_name', 'Date', 'ticker']: duplicate_rows=0, duplicate_groups=0
approved_signal_long duplicate diagnostics for ['signal_name', 'Date', 'ticker']: duplicate_rows=0, duplicate_groups=0
approved_signal_long duplicate diagnostics for ['signal_name', 'Date', 'ticker']: duplicate_rows=0, duplicate_groups=0
approved_signal_long duplicate diagnostics for ['signal_name', 'Date', 'ticker']: duplicate_rows=0, duplicate_groups=0
approved_signal_long duplicate diagnostics for [

ValueError: Cannot pivot signal_name 'expanded_reversal_1d' because duplicate Date/ticker rows were found: duplicate_rows=1,002,844, duplicate_groups=478. Use duplicate_policy='last' for temporary recovery or duplicate_policy='mean' only when averaging duplicates is intended.

## 8. Build decay summary

In [ ]:
decay_summary_enriched = decay_summary.merge(
    signal_best_horizon[['signal_name', 'signal_family', 'best_horizon', 'signal_direction', 'signal_strength']],
    on='signal_name',
    how='left',
)

interpretability_columns = [
    'signal_name',
    'horizon',
    'mean_rolling_ic',
    'recent_ic',
    'early_ic',
    'ic_change',
    'decay_slope',
    'sign_stability',
    'decay_status',
    'decay_risk_flag',
]

status_counts = decay_summary_enriched['decay_status'].value_counts()
risk_counts = decay_summary_enriched['decay_risk_flag'].value_counts()
top_stable_signals = (
    decay_summary_enriched.loc[decay_summary_enriched['decay_status'].eq('STABLE')]
    .sort_values(['mean_rolling_ic', 'sign_stability'], ascending=[False, False])
    .head(15)
)
strongest_decaying_signals = (
    decay_summary_enriched.loc[decay_summary_enriched['decay_status'].eq('DECAYING')]
    .sort_values(['decay_slope', 'ic_change'])
    .head(15)
)

print('Decay summary sample')
display(decay_summary_enriched[interpretability_columns].head())

print('Decay status counts')
display(status_counts.rename('signal_horizon_count'))

print('Decay risk counts')
display(risk_counts.rename('signal_horizon_count'))


Decay summary sample


,signal_name,horizon,mean_rolling_ic,recent_ic,early_ic,ic_change,decay_slope,sign_stability,decay_status,decay_risk_flag
0,close_position_reversal_5,1,0.006179,0.003088,0.008711,-0.005624,-7.642554e-06,0.659919,STABLE,LOW_DECAY_RISK
1,close_position_reversal_5,5,0.007342,-0.009125,0.017883,-0.027008,-2.528297e-05,0.588235,UNSTABLE,MODERATE_DECAY_RISK
2,close_position_reversal_5,10,0.005600,-0.014145,0.016085,-0.030229,-3.075634e-05,0.544992,UNSTABLE,MODERATE_DECAY_RISK
3,close_position_reversal_5,20,0.008028,-0.029568,0.023417,-0.052986,-4.790490e-05,0.538068,UNSTABLE,MODERATE_DECAY_RISK
4,dollar_volume_shock_20,1,0.003069,0.000210,0.003710,-0.003499,-6.565470e-07,0.639487,STABLE,LOW_DECAY_RISK


Decay status counts


decay_status
UNSTABLE    33
STABLE      27
Name: signal_horizon_count, dtype: int64

Decay risk counts


decay_risk_flag
MODERATE_DECAY_RISK    34
LOW_DECAY_RISK         24
HIGH_DECAY_RISK         2
Name: signal_horizon_count, dtype: int64

## 9. Save outputs to SQLite

In [ ]:
saved_paths = save_signal_decay_outputs(
    decay_curve=decay_curve,
    decay_summary=decay_summary_enriched,
    db_path=sqlite_db_path,
    run_id=run_id,
    decay_version=DECAY_VERSION,
)

sqlite_tables_written = pd.DataFrame(
    [
        {
            'artifact': artifact,
            'current_table': tables[0],
            'history_table': tables[1],
            'sqlite_path': str(saved_paths[artifact]),
        }
        for artifact, tables in SIGNAL_DECAY_TABLES.items()
    ]
)

display(sqlite_tables_written)

,artifact,current_table,history_table,sqlite_path
0,curve,signal_decay_curve_current,signal_decay_curve_history,/Users/AnyiXu_1/Desktop/multi-factor-equity-al...
1,summary,signal_decay_summary_current,signal_decay_summary_history,/Users/AnyiXu_1/Desktop/multi-factor-equity-al...


## 10. Final display

In [ ]:
print('Decay status counts')
display(status_counts.rename('signal_horizon_count'))

print('Decay risk counts')
display(risk_counts.rename('signal_horizon_count'))

print('Top stable signals')
display(top_stable_signals[interpretability_columns])

print('Strongest decaying signals')
display(strongest_decaying_signals[interpretability_columns])

print('Rolling IC curve sample')
display(decay_curve.dropna(subset=['rolling_ic']).head(25))

print('SQLite tables written')
display(sqlite_tables_written)


Decay status counts


decay_status
UNSTABLE    33
STABLE      27
Name: signal_horizon_count, dtype: int64

Decay risk counts


decay_risk_flag
MODERATE_DECAY_RISK    34
LOW_DECAY_RISK         24
HIGH_DECAY_RISK         2
Name: signal_horizon_count, dtype: int64

Top stable signals


...

,signal_name,horizon,mean_rolling_ic,recent_ic,early_ic,ic_change,decay_slope,sign_stability,decay_status,decay_risk_flag


Rolling IC curve sample


,Date,rolling_ic,signal_name,horizon,method,window
121,2018-06-26,0.005067,close_position_reversal_5,1,spearman,63
122,2018-06-27,-0.000764,close_position_reversal_5,1,spearman,63
123,2018-06-28,0.005040,close_position_reversal_5,1,spearman,63
124,2018-06-29,0.006000,close_position_reversal_5,1,spearman,63
125,2018-07-02,-0.000066,close_position_reversal_5,1,spearman,63
126,2018-07-03,-0.000071,close_position_reversal_5,1,spearman,63
127,2018-07-05,0.000504,close_position_reversal_5,1,spearman,63
128,2018-07-06,0.001503,close_position_reversal_5,1,spearman,63
129,2018-07-09,-0.000673,close_position_reversal_5,1,spearman,63
130,2018-07-10,-0.002743,close_position_reversal_5,1,spearman,63


SQLite tables written


,artifact,current_table,history_table,sqlite_path
0,curve,signal_decay_curve_current,signal_decay_curve_history,/Users/AnyiXu_1/Desktop/multi-factor-equity-al...
1,summary,signal_decay_summary_current,signal_decay_summary_history,/Users/AnyiXu_1/Desktop/multi-factor-equity-al...


In [ ]:
decay = load_table("signal_decay_summary_current")

# 1. No missing critical metrics
decay[[
    "mean_rolling_ic",
    "recent_ic",
    "early_ic",
    "decay_slope",
    "sign_stability"
]].isna().sum()

# 2. Sign stability must be in [0,1]
(decay["sign_stability"].between(0, 1)).all()

# 3. No zero-length rolling samples
(decay["n_rolling_obs"] > 0).all()

# 4. Direction consistency sanity
(decay["recent_ic"] * decay["signal_direction"].map({
    "POSITIVE_EDGE": 1,
    "NEGATIVE_EDGE_REVERSE_SIGNAL": -1
}) > 0).mean()

np.float64(0.48333333333333334)

In [ ]:
decay.sort_values("mean_rolling_ic", ascending=False).head(10)
decay.sort_values("decay_slope").head(10)
decay.sort_values("sign_stability").head(10)

,signal_name,horizon,n_rolling_obs,mean_rolling_ic,ic_volatility,rolling_ic_min,rolling_ic_max,rolling_ic_recent_252d_mean,decay_slope,abs_decay_slope,...,min_rolling_ic_drawdown,half_life_proxy,decay_status,decay_risk_flag,signal_family,best_horizon,signal_direction,signal_strength,run_id,decay_version
56,vol_surprise_20_60,1,1950,-0.000053,0.013259,-0.042858,0.040732,-0.003677,-4.088122e-06,4.088122e-06,...,-0.083590,12.989496,UNSTABLE,HIGH_DECAY_RISK,volatility_structure,20,POSITIVE_EDGE,NO_SIGNAL,phase2_nb03c_signal_decay_20260508_103600,phase2_signal_decay_v1
59,vol_surprise_20_60,20,1931,0.006489,0.057639,-0.141983,0.193171,-0.001066,-1.639391e-05,1.639391e-05,...,-0.335154,395.819354,UNSTABLE,HIGH_DECAY_RISK,volatility_structure,20,POSITIVE_EDGE,NO_SIGNAL,phase2_nb03c_signal_decay_20260508_103600,phase2_signal_decay_v1
7,dollar_volume_shock_20,20,1931,-0.000603,0.025217,-0.059322,0.086139,-0.017260,-3.384728e-06,3.384728e-06,...,-0.145461,178.136574,UNSTABLE,MODERATE_DECAY_RISK,liquidity_flow,1,POSITIVE_EDGE,NO_SIGNAL,phase2_nb03c_signal_decay_20260508_103600,phase2_signal_decay_v1
43,relative_return_zscore_60,20,1902,0.000172,0.061983,-0.175458,0.204689,-0.009242,-2.747054e-05,2.747054e-05,...,-0.380147,6.258409,UNSTABLE,MODERATE_DECAY_RISK,cross_sectional_relative_value,10,NEGATIVE_EDGE_REVERSE_SIGNAL,NO_SIGNAL,phase2_nb03c_signal_decay_20260508_103600,phase2_signal_decay_v1
51,three_day_overextension_reversal,20,1971,-0.001187,0.044469,-0.139642,0.130494,-0.020389,-1.657007e-05,1.657007e-05,...,-0.270136,71.628154,UNSTABLE,MODERATE_DECAY_RISK,true_short_term_reversal,1,POSITIVE_EDGE,NO_SIGNAL,phase2_nb03c_signal_decay_20260508_103600,phase2_signal_decay_v1
22,overnight_gap_reversal_1,10,1990,-0.003466,0.033419,-0.098034,0.089141,-0.009026,-2.097346e-07,2.097346e-07,...,-0.160191,16524.323587,UNSTABLE,MODERATE_DECAY_RISK,true_short_term_reversal,20,NEGATIVE_EDGE_REVERSE_SIGNAL,NO_SIGNAL,phase2_nb03c_signal_decay_20260508_103600,phase2_signal_decay_v1
58,vol_surprise_20_60,10,1941,0.004886,0.041910,-0.123960,0.130468,-0.014341,-1.503572e-05,1.503572e-05,...,-0.254428,324.957100,UNSTABLE,MODERATE_DECAY_RISK,volatility_structure,20,POSITIVE_EDGE,NO_SIGNAL,phase2_nb03c_signal_decay_20260508_103600,phase2_signal_decay_v1
5,dollar_volume_shock_20,5,1946,-0.000417,0.018025,-0.060916,0.054188,-0.006160,-8.540881e-06,8.540881e-06,...,-0.115104,48.795426,UNSTABLE,MODERATE_DECAY_RISK,liquidity_flow,1,POSITIVE_EDGE,NO_SIGNAL,phase2_nb03c_signal_decay_20260508_103600,phase2_signal_decay_v1
6,dollar_volume_shock_20,10,1941,0.001824,0.020498,-0.046393,0.054142,0.002607,-1.112902e-06,1.112902e-06,...,-0.099658,1639.283702,UNSTABLE,MODERATE_DECAY_RISK,liquidity_flow,1,POSITIVE_EDGE,NO_SIGNAL,phase2_nb03c_signal_decay_20260508_103600,phase2_signal_decay_v1
24,price_impact_proxy_20,1,1950,-0.000758,0.014536,-0.044587,0.035768,-0.003543,-7.022254e-06,7.022254e-06,...,-0.080355,108.013666,UNSTABLE,MODERATE_DECAY_RISK,liquidity_flow,20,POSITIVE_EDGE,NO_SIGNAL,phase2_nb03c_signal_decay_20260508_103600,phase2_signal_decay_v1


In [ ]:
decay["mean_rolling_ic"].describe()
decay["sign_stability"].describe()
decay["decay_slope"].describe()

count    60.000000
mean     -0.000003
std       0.000016
min      -0.000048
25%      -0.000013
50%      -0.000003
75%       0.000004
max       0.000033
Name: decay_slope, dtype: float64